# 📚 Research Literature Review Agent

**Automatically generate a publication-quality literature review from arXiv papers.**

```
User Query → arXiv Search → Embedding + FAISS → LLM Summarization → Synthesis → Review
```

---
### 🚦 Before you begin
1. Add your OpenAI API key to **Colab Secrets** (🔑 icon in left sidebar)  
   Name: `OPENAI_API_KEY` | Value: `sk-...`
2. Run cells **in order** (Shift+Enter)
3. Change `TOPIC` and `MAX_PAPERS` in Step 4 to customise your review

---
## Step 1 — Install Dependencies
*Takes ~60 seconds on first run*

In [ ]:
!pip install -q \
    langchain langchain-community langchain-openai \
    openai arxiv faiss-cpu tiktoken python-dotenv
print('✅ Packages installed')

---
## Step 2 — Load API Key from Colab Secrets

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('✅ API key loaded from Colab secrets.')
except Exception as e:
    print(f'⚠ Could not load from secrets ({e}).')
    print('  Set manually: os.environ["OPENAI_API_KEY"] = "sk-..."')

---
## Step 3 — Clone Repository

In [ ]:
# Replace with your actual GitHub URL after pushing
!git clone https://github.com/aliakarma/research-literature-agent.git
%cd research-literature-agent
import sys; sys.path.insert(0, '.')
print('✅ Repository cloned and path configured.')

---
## Step 4 — Configure and Run the Agent

⚙️ **Edit `TOPIC` and `MAX_PAPERS` below to customise your review.**

In [ ]:
# ════════════════════════════════════════
#  CONFIGURE YOUR SEARCH HERE
TOPIC      = 'Agentic AI for climate prediction'   # ← change this
MAX_PAPERS = 8                                      # ← 5–15 recommended
SAVE_LATEX = False                                  # ← True to get a .tex file
# ════════════════════════════════════════

from research_agent import run_literature_review_agent

review = run_literature_review_agent(
    topic=TOPIC,
    max_papers=MAX_PAPERS,
    save_txt=True,
    save_latex=SAVE_LATEX,
    output_dir='outputs',
)

---
## Step 5 — Download the Output File

In [ ]:
import glob
from google.colab import files

output_files = sorted(glob.glob('outputs/*.txt') + glob.glob('outputs/*.tex'))
if output_files:
    latest = output_files[-1]
    print(f'Downloading: {latest}')
    files.download(latest)
else:
    print('No output files found. Check that save_txt=True.')

---
## Step 6 — Inspect Individual Pipeline Stages (Optional)
Run these cells to see what happens at each step.

In [ ]:
# ── Stage 1: Fetch papers ─────────────────────────────────────────────────────
from research_agent.config import Config
from research_agent.arxiv_retriever import ArxivRetriever

config = Config(max_papers=5)
retriever = ArxivRetriever(config)
papers = retriever.search(TOPIC)

print(f'\nFirst paper:')
print(f'  Title   : {papers[0]["title"]}')
print(f'  Authors : {papers[0]["authors"][:2]}')
print(f'  Date    : {papers[0]["date"]}')
print(f'  Abstract: {papers[0]["abstract"][:250]}...')

In [ ]:
# ── Stage 2: Summarize one paper ──────────────────────────────────────────────
from research_agent.summarizer import Summarizer

summarizer = Summarizer(config)
summary = summarizer.summarize_paper(papers[0])
print('SUMMARY:')
print(summary)

In [ ]:
# ── Stage 3: Test vector store similarity search ──────────────────────────────
from research_agent.arxiv_retriever import ArxivRetriever
from research_agent.vector_store import VectorStore

doc_texts = ArxivRetriever.to_plain_text(papers)
vs = VectorStore(config)
vs.build(doc_texts)

chunks = vs.retrieve('deep learning atmospheric model', k=2)
for i, c in enumerate(chunks, 1):
    print(f'\n--- Chunk {i} ---')
    print(c[:300])